# Packages/libraries installation


In [ ]:
%pip install numpy pandas scikit-learn xgboost imbalanced-learn

In [2]:
import numpy as np
import pandas as pd
import os
from datetime import datetime

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OneHotEncoder
from sklearn.discriminant_analysis import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Data preparation


In [3]:
# Constants
DATA_PATH = "data/"
RANDOM_SEED = 3

In [4]:
def scale_data(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
    if len(X_val) > 1:
        X_val_scaled = scaler.transform(X_val)
        X_val = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
    return X_train, X_val, X_test

In [5]:
def decompose_data(
    X_train,
    X_val,
    X_test,
    n_components=25,
    random_state=RANDOM_SEED,
    log_steps=True,
):
    decomposer = TruncatedSVD(n_components=n_components, random_state=random_state)
    if log_steps:
        print("Decomposing data...")

    X_train_reduced = decomposer.fit_transform(X_train)
    if len(X_val) > 1:
        X_val_reduced = decomposer.transform(X_val)
    else:
        X_val_reduced = X_val
    X_test_reduced = decomposer.transform(X_test)

    X_train_reduced = pd.DataFrame(X_train_reduced, index=X_train.index)
    X_val_reduced = pd.DataFrame(X_val_reduced, index=X_val.index)
    X_test_reduced = pd.DataFrame(X_test_reduced, index=X_test.index)
    X_train_reduced.columns = X_train_reduced.columns.astype(str)
    X_val_reduced.columns = X_val_reduced.columns.astype(str)
    X_test_reduced.columns = X_test_reduced.columns.astype(str)
    return X_train_reduced, X_val_reduced, X_test_reduced

In [6]:
def load_X_data(random_state=RANDOM_SEED, split_size=0.2, log_steps=True):

    train_data = np.load(f"{DATA_PATH}train.npz")
    X = train_data["X_train"]
    X = pd.DataFrame(X)
    test_data = np.load(f"{DATA_PATH}test.npz")
    X_test = test_data["X_test"]
    X_test = pd.DataFrame(X_test)
    y = train_data["y_train"]

    if split_size > 0:
        X_train, X_val = train_test_split(
            X,
            test_size=split_size,
            random_state=random_state,
            stratify=y,
        )
    else:
        X_train = X
        X_val = pd.DataFrame()

    return X_train, X_val, X_test

In [7]:
def load_y_data(random_state=RANDOM_SEED, split_size=0.2, log_steps=True):
    path = f"{DATA_PATH}/y_splits/random_state_{random_state}_split_{split_size}/"
    if os.path.isdir(path):
        if log_steps:
            print("Loading y data from disk...")
        y_train = pd.read_csv(f"{path}y_train.csv", index_col=[0])
        y_val = pd.read_csv(f"{path}y_val.csv", index_col=[0])
        return y_train, y_val

    if log_steps:
        print("Loading and splitting y data...")
    if os.path.isdir(f"{DATA_PATH}y_train.csv"):
        y = pd.read_csv(f"{DATA_PATH}y_train.csv")
    else:
        train_data = np.load(f"{DATA_PATH}train.npz")
        y = train_data["y_train"]
        y = pd.DataFrame(y, columns=["label"])
        y.to_csv(f"{DATA_PATH}y_train.csv", index=False)

    if split_size > 0:
        y_train, y_val = train_test_split(
            y,
            test_size=split_size,
            random_state=random_state,
            stratify=y,
        )
    else:
        y_train = y
        y_val = []
    os.makedirs(path, exist_ok=True)
    y_train = pd.DataFrame(y_train, columns=["label"])
    y_val = pd.DataFrame(y_val, columns=["label"])
    y_train.to_csv(f"{path}y_train.csv")
    y_val.to_csv(f"{path}y_val.csv")

    return y_train, y_val

In [8]:
def get_X_data(
    n_components=25,
    random_state=RANDOM_SEED,
    split_size=0.2,
    log_steps=True,
):
    folder_path = f"{DATA_PATH}X_processed_TruncatedSVD_random_state_{random_state}_split_{split_size}/{n_components}/"
    if os.path.isdir(folder_path):
        X_train = pd.read_csv(f"{folder_path}X_train_processed.csv", index_col=[0])
        X_val = pd.read_csv(f"{folder_path}X_val_processed.csv", index_col=[0])
        X_test = pd.read_csv(f"{folder_path}X_test_processed.csv", index_col=[0])
        return X_train, X_val, X_test

    X_train, X_val, X_test = load_X_data(
        random_state=random_state, split_size=split_size
    )

    X_train, X_val, X_test = decompose_data(
        X_train,
        X_val,
        X_test,
        n_components=n_components,
        random_state=random_state,
        log_steps=log_steps,
    )

    os.makedirs(folder_path, exist_ok=True)
    X_train = pd.DataFrame(X_train, index=X_train.index)
    X_train.to_csv(f"{folder_path}X_train_processed.csv")
    X_val = pd.DataFrame(X_val, index=X_val.index)
    X_val.to_csv(f"{folder_path}X_val_processed.csv")
    X_test = pd.DataFrame(X_test, index=X_test.index)
    X_test.to_csv(f"{folder_path}X_test_processed.csv")
    return X_train, X_val, X_test

In [9]:
def load_data(
    n_components=25,
    random_state=RANDOM_SEED,
    split_size=0.2,
    log_steps=True,
):
    y_train, y_val = load_y_data(
        random_state=random_state, split_size=split_size, log_steps=log_steps
    )
    X_train, X_val, X_test = get_X_data(
        n_components=n_components,
        random_state=random_state,
        split_size=split_size,
        log_steps=log_steps,
    )

    return X_train, y_train, X_val, y_val, X_test

In [10]:
def preprocess_metadata(
    metadata_train,
    metadata_val,
    metadata_test,
    print_infos=True,
):
    # fix formatting for further methods
    metadata_train = metadata_train.apply(
        lambda col: (
            col.str.lower().str.replace(r"[^a-z0-9_]", "", regex=True)
            if col.dtype == "object"
            else col
        )
    )
    metadata_val = metadata_val.apply(
        lambda col: (
            col.str.lower().str.replace(r"[^a-z0-9_]", "", regex=True)
            if col.dtype == "object"
            else col
        )
    )
    metadata_test = metadata_test.apply(
        lambda col: (
            col.str.lower().str.replace(r"[^a-z0-9_]", "", regex=True)
            if col.dtype == "object"
            else col
        )
    )

    for col in metadata_train.columns:
        # Setting missing values to column mode
        mode_value = metadata_train[col].mode()[0]
        if metadata_train[col].isnull().values.any():
            if print_infos:
                print(f"metadata_train contains missing values for {col}")
            metadata_train[col] = metadata_train[col].fillna(mode_value)
        if len(metadata_val) > 0 and metadata_val[col].isnull().values.any():
            if print_infos:
                print(f"metadata_val contains missing values for {col}")
            metadata_val[col] = metadata_val[col].fillna(mode_value)
        if metadata_test[col].isnull().values.any():
            if print_infos:
                print(f"metadata_test contains missing values for {col}")
            metadata_test[col] = metadata_test[col].fillna(mode_value)

        # Setting rare categoriesto mode
        counts = metadata_train[col].value_counts()
        rare_categories = counts[counts < 20].index
        metadata_train[col] = metadata_train[col].where(
            ~metadata_train[col].isin(rare_categories), mode_value
        )
        selected_categories = metadata_train[col].unique()

        if len(metadata_val) > 0:
            metadata_val[col] = metadata_val[col].where(
                metadata_val[col].isin(selected_categories), mode_value
            )
        metadata_test[col] = metadata_test[col].where(
            metadata_test[col].isin(selected_categories), mode_value
        )

    # One-hot encoding categorical data
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoder.fit(metadata_train)
    metadata_train = pd.DataFrame(
        encoder.transform(metadata_train),
        columns=encoder.get_feature_names_out(),
        index=metadata_train.index,
    )
    if len(metadata_val) > 0:
        metadata_val = pd.DataFrame(
            encoder.transform(metadata_val),
            columns=encoder.get_feature_names_out(),
            index=metadata_val.index,
        )
    metadata_test = pd.DataFrame(
        encoder.transform(metadata_test),
        columns=encoder.get_feature_names_out(),
        index=metadata_test.index,
    )

    return metadata_train, metadata_val, metadata_test

In [11]:
def load_metadata(train_indexes, print_infos=True):
    metadata_train = pd.read_csv(f"{DATA_PATH}metadata_train.csv")
    metadata_test = pd.read_csv(f"{DATA_PATH}metadata_test.csv")

    if print_infos:
        for col in metadata_train.columns:
            print(f"Column: {col}")
            print(metadata_train[col].value_counts())

        for col in metadata_test.columns:
            print(f"Column: {col}")
            print(metadata_test[col].value_counts())

    # columns and "Unnamed: 0", "ID" are irrelevant and "Organism group" has only one value
    # columns "Create date", "Laboratory typing platform", "Isolation type" and "Location" are not biologically relevant
    metadata_train.drop(
        columns=[
            "Unnamed: 0",
            "ID",
            "Organism group",
            "Create date",
            "Laboratory typing platform",
            "Isolation type",
            "Location",
        ],
        inplace=True,
    )
    metadata_test.drop(
        columns=[
            "Unnamed: 0",
            "ID",
            "Organism group",
            "Create date",
            "Laboratory typing platform",
            "Isolation type",
            "Location",
        ],
        inplace=True,
    )
    val_indexes = metadata_train.index.difference(train_indexes)
    metadata_val = metadata_train.loc[val_indexes]
    metadata_train = metadata_train.loc[train_indexes]
    return metadata_train, metadata_val, metadata_test

In [12]:
def get_prepared_data(
    n_components=25,
    add_metadata=True,
    random_state=RANDOM_SEED,
    split_size=0.2,
    log_steps=True,
):
    if log_steps:
        print("Loading data...")
    X_train, y_train, X_val, y_val, X_test = load_data(
        n_components=n_components,
        random_state=random_state,
        split_size=split_size,
        log_steps=log_steps,
    )

    if log_steps:
        print("Scaling data...")
    X_train, X_val, X_test = scale_data(X_train, X_val, X_test)

    if add_metadata:
        if log_steps:
            print("Loading metadata...")
        metadata_train, metadata_val, metadata_test = load_metadata(
            y_train.index, print_infos=False
        )
        if log_steps:
            print("Preprocessing metadata...")
        metadata_train, metadata_val, metadata_test = preprocess_metadata(
            metadata_train,
            metadata_val,
            metadata_test,
            print_infos=False,
        )

        X_train = pd.concat([X_train, metadata_train], axis=1)
        X_val = pd.concat([X_val, metadata_val], axis=1)
        X_test = pd.concat([X_test, metadata_test], axis=1)

    return X_train, y_train, X_val, y_val, X_test

# Model training and prediction


In [13]:
def train_model(pipeline, X_train, y_train, X_val, y_val):
    pipeline.fit(X_train, y_train)
    y_val_pred = pipeline.predict(X_val)
    accuracy = accuracy_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred, average="macro")
    print(
        f"Model: {pipeline.named_steps['model'].__class__.__name__}, Accuracy: {accuracy}, F1 Score: {f1}"
    )
    return pipeline, y_val_pred

In [14]:
def combine_multiple_models_predictions(predictions):
    values = np.array(predictions).T
    majority = (values.mean(axis=1) >= 0.5).astype(int)
    y_pred = pd.DataFrame(majority, columns=["label"])

    return y_pred

In [15]:
def train_and_predict(
    models,  # array of model classes and their args
    n_components=[250],
    samplers=[None],
    add_metadatas=[False],
    generate_file=False,
    random_state=RANDOM_SEED,
):

    oof_preds = []
    for i in range(5):

        y_preds = []
        y_val_preds = []
        for model_class, model_arg, n_component, sampler, add_metadata in zip(
            [m[0] for m in models],
            [m[1] for m in models],
            n_components,
            samplers,
            add_metadatas,
        ):
            X_train, y_train, X_val, y_val, X_predict = get_prepared_data(
                n_components=n_component,
                add_metadata=add_metadata,
                random_state=i,
                split_size=0.2,
                log_steps=False,
            )
            y_train = y_train.values.ravel()
            y_val = y_val.values.ravel()
            print(f"Training model: {model_class.__name__}")
            pipeline = Pipeline(
                [("sampler", sampler), ("model", model_class(**model_arg))]
            )
            model_trained, y_val_pred = train_model(
                pipeline, X_train, y_train, X_val, y_val
            )
            y_pred = model_trained.predict(X_predict)

            y_preds.append(y_pred)
            y_val_preds.append(y_val_pred)

        y_val_pred = combine_multiple_models_predictions(y_val_preds)
        accuracy = accuracy_score(y_val, y_val_pred)
        f1 = f1_score(y_val, y_val_pred, average="macro")
        print(f"Combined Model - Accuracy: {accuracy}, F1 Score: {f1}\n")

        y_pred = combine_multiple_models_predictions(y_preds)
        oof_preds.append(y_pred)

    # final submission prediction
    y_pred = combine_multiple_models_predictions(np.array(oof_preds).reshape(5, -1))

    if generate_file:
        metadata_df = pd.read_csv("data/metadata_test.csv")[["ID"]]
        temp = pd.concat([metadata_df, y_pred], axis=1)
        temp.rename(columns={"ID": "id"}, inplace=True)
        counts = temp["label"].value_counts()
        print(counts)
        os.makedirs("predictions", exist_ok=True)
        pd.DataFrame(temp[["id", "label"]]).to_csv(
            f"predictions/predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{random_state}.csv",
            index=False,
        )
    return y_pred

In [16]:
RANDOM_STATE = 2031867962
print(f"Using random state: {RANDOM_STATE}")

models = [
    (
        ExtraTreesClassifier,
        {
            "max_depth": None,
            "min_samples_split": 5,
            "min_samples_leaf": 2,
            "criterion": "gini",
            "random_state": RANDOM_STATE,
            "n_estimators": 200,
        },
    ),
    (
        ExtraTreesClassifier,
        {
            "max_depth": None,
            "min_samples_split": 5,
            "min_samples_leaf": 2,
            "criterion": "gini",
            "random_state": RANDOM_STATE,
            "n_estimators": 200,
        },
    ),
    (
        ExtraTreesClassifier,
        {
            "max_depth": None,
            "min_samples_split": 10,
            "min_samples_leaf": 2,
            "criterion": "entropy",
            "random_state": RANDOM_STATE,
            "n_estimators": 200,
        },
    ),
    (
        MLPClassifier,
        {
            "activation": "relu",
            "random_state": RANDOM_STATE,
            "alpha": 0.0001,
            "hidden_layer_sizes": (100, 50),
            "learning_rate": "invscaling",
            "learning_rate_init": 0.01,
            "solver": "adam",
            "max_iter": 2000,
            "early_stopping": True,
            "alpha": 1e-4,
        },
    ),
    (
        MLPClassifier,
        {
            "activation": "relu",
            "random_state": RANDOM_STATE,
            "alpha": 0.0001,
            "hidden_layer_sizes": (100, 50),
            "learning_rate": "invscaling",
            "learning_rate_init": 0.01,
            "solver": "adam",
            "max_iter": 2000,
            "early_stopping": True,
            "alpha": 1e-4,
        },
    ),
    (
        XGBClassifier,
        {
            "random_state": RANDOM_STATE,
            "learning_rate": 0.2,
            "max_depth": 5,
            "n_estimators": 100,
            "eval_metric": "logloss",
        },
    ),
    (
        XGBClassifier,
        {
            "random_state": RANDOM_STATE,
            "learning_rate": 0.2,
            "max_depth": 5,
            "n_estimators": 100,
            "eval_metric": "logloss",
        },
    ),
    (
        SVC,
        {
            "random_state": RANDOM_STATE,
            "C": 5,
            "kernel": "poly",
            "gamma": "scale",
            "coef0": 0.1,
            "class_weight": None,
            "degree": 2,
        },
    ),
    (
        SVC,
        {
            "random_state": RANDOM_STATE,
            "C": 5,
            "kernel": "poly",
            "gamma": "scale",
            "coef0": 0.1,
            "class_weight": None,
            "degree": 2,
        },
    ),
]

n_components = [100, 100, 100, 200, 200, 100, 100, 200, 200]
samplers = [
    SMOTE(k_neighbors=3, random_state=RANDOM_STATE),
    SMOTE(k_neighbors=3, random_state=RANDOM_STATE),
    SMOTE(k_neighbors=7, random_state=RANDOM_STATE),
    SMOTE(k_neighbors=20, random_state=RANDOM_STATE),
    SMOTE(k_neighbors=20, random_state=RANDOM_STATE),
    SMOTE(k_neighbors=50, random_state=RANDOM_STATE),
    SMOTE(k_neighbors=50, random_state=RANDOM_STATE),
    SMOTE(k_neighbors=50, random_state=RANDOM_STATE),
    SMOTE(k_neighbors=50, random_state=RANDOM_STATE),
]

add_metadatas = [
    True,
    False,
    True,
    True,
    False,
    True,
    False,
    True,
    False,
]
if (
    len(models) != len(n_components)
    or len(models) != len(samplers)
    or len(models) != len(add_metadatas)
):
    print(len(models), len(n_components), len(samplers), len(add_metadatas))
    raise ValueError(
        "Models, n_components, samplers, and add_metadatas must have the same length."
    )

train_and_predict(
    models=models,
    n_components=n_components,
    samplers=samplers,
    add_metadatas=add_metadatas,
    generate_file=True,
    random_state=RANDOM_STATE,
)

Using random state: 2031867962
Training model: ExtraTreesClassifier
Model: ExtraTreesClassifier, Accuracy: 0.9020618556701031, F1 Score: 0.7741421568627451
Training model: ExtraTreesClassifier
Model: ExtraTreesClassifier, Accuracy: 0.8865979381443299, F1 Score: 0.7430618265005117
Training model: ExtraTreesClassifier
Model: ExtraTreesClassifier, Accuracy: 0.8994845360824743, F1 Score: 0.7886090272830142
Training model: MLPClassifier
Model: MLPClassifier, Accuracy: 0.8943298969072165, F1 Score: 0.781191972820929
Training model: MLPClassifier
Model: MLPClassifier, Accuracy: 0.8969072164948454, F1 Score: 0.7913080895008606
Training model: XGBClassifier
Model: XGBClassifier, Accuracy: 0.8994845360824743, F1 Score: 0.7918655351223471
Training model: XGBClassifier
Model: XGBClassifier, Accuracy: 0.884020618556701, F1 Score: 0.766948304168502
Training model: SVC
Model: SVC, Accuracy: 0.9072164948453608, F1 Score: 0.8121772805507745
Training model: SVC
Model: SVC, Accuracy: 0.904639175257732, F

,label
0,0
1,0
2,0
3,0
4,1
...,...
1087,0
1088,0
1089,0
1090,1
